In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage

In [3]:
basic_model = init_chat_model(model="gpt-5-nano")
advanced_model = init_chat_model(model="gpt-5")

In [4]:
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:

    print("middleware  실행 : dynamic_model_selection")

    message_len = len(request.state["messages"])

    if message_len > 10:
        model = advanced_model
    else:
        model = basic_model

    return handler(request.override(model=model))

In [5]:
agent = create_agent(model=basic_model, tools=[], middleware=[dynamic_model_selection])

In [6]:
agent.invoke({"messages": HumanMessage(content="RAG가 무엇인지 설명해주세요.")})

middleware  실행 : dynamic_model_selection


{'messages': [HumanMessage(content='RAG가 무엇인지 설명해주세요.', additional_kwargs={}, response_metadata={}, id='e897b7f7-ebc6-44d9-b83d-2eb9ed8a28c8'),
  AIMessage(content='RAG(Retrieval-Augmented Generation)는 생성 모델이 외부 지식 문서를 검색해 그 정보를 바탕으로 답을 생성하도록 하는 접근 방식입니다.\n\n주요 내용\n\n- 구성 요소\n  - Retriever(검색기): 쿼리와 관련된 문서를 벡터 공간에서 찾아 상위 k개를 반환합니다. 일반적으로 Dense Passage Retrieval(DPR) 같은 임베딩 기반 검색이 사용됩니다.\n  - Generator(생성기): 검색된 문서들을 입력으로 받아 질문에 대한 답을 생성합니다. BART, T5 같은 시퀀스-투-시퀀스 모델이 많이 쓰입니다.\n  - (일부 구현에서) 문서를 한 번에 인코딩하는 RAG-Sequence와 토큰 단위로 문서를 활용하는 RAG-Token와 같은 변형이 있습니다.\n\n- 작동 원리(간단한 흐름)\n  1) 사용자의 질의가 들어오면Retriever가 관련 문서들을 검색합니다.\n  2) 검색된 문서들을 Generator의 입력과 함께 제공합니다.\n  3) 생성기는 이 문서들에 기반해 답변을 생성합니다. 일부 구현은 문서를 디코딩 동안 매 토큰마다 참조하기도 합니다.\n  4) 필요하면 재검색이나 재정렬로 품질을 높이기도 합니다.\n\n- 왜 사용하나요?\n  - 최신 지식이나 특정 도메인 문서를 외부에서 참조해 답을 만들 수 있어, 순수하게 파라미터 안에 저장된 지식만으로 생성하는 것보다 사실성(factuality)을 높일 수 있습니다.\n  - 긴-tail 지식이나 넓은 범위의 정보에 대응하기 좋습니다.\n  - 대화형 시스템에서 출처를 문서로 연결해 근거를 제시하기 쉽습니다.\n\n- 장점\n  - 사실성 향상 및 외부 지식

In [8]:
@dataclass
class UserContext:
    user_id: str

In [9]:
store = InMemoryStore()

In [10]:
@tool
def save_preference(
    category: str, value: str, runtime: ToolRuntime[UserContext]
) -> str:
    """
    사용자의 선호 정보를 저장합니다.
    예)  category='drink', values='tea'
    """
    user_id = runtime.context.user_id
    namespace = ("users", user_id, "preference")

    runtime.store.put(namespace, category, {"value": value})

    return f"사용자 선호 정보가 저장되었습니다."


@tool
def load_preference(category: str, runtime: ToolRuntime[UserContext]) -> str:
    """사용자의 선호 정보를 불러옵니다."""

    user_id = runtime.context.user_id
    namespace = ("users", user_id, "preference")

    memory = runtime.store.get(namespace, category)

    if memory is None:
        return f"{category}에 대한 저장된 정보가 없습니다."

    return f"{category}에 대한 정보는 {memory.value['value']} 입니다."


@tool
def recommend_by_preference(runtime: ToolRuntime[UserContext]) -> str:
    """저장된 선호를 바탕으로 간단한 추천을 합니다."""

    user_id = runtime.context.user_id
    namespace = ("users", user_id, "preference")

    drink_memory = runtime.store.get(namespace, "drink")
    food_memory = runtime.store.get(namespace, "food")

    drink = drink_memory.value["value"] if drink_memory else None
    food = food_memory.value["value"] if food_memory else None

    if drink and food:
        return (
            f"당신은 {drink}와 {food}를 좋아하므로, {food}와 함께 {drink}를 추천합니다."
        )
    elif drink:
        return f"당신은 {drink}를 좋아하므로, 오늘은 {drink}를 추천합니다."
    elif food:
        return f"당신은 {food}를 좋아하므로, {food} 관련 메뉴를 추천합니다."
    else:
        return "저장된 선호 정보가 없어서 아직 추천하기 어렵습니다."

In [11]:
@wrap_tool_call
def log_tool_calls(request, handler):

    print(f"[TOOL START]  {request.tool.name} ")

    tool_call = getattr(request, "tool_call", None)
    if tool_call and "args" in tool_call and tool_call["args"] is not None:
        print(f"[TOOL ARGS] {tool_call['args']}")

    try:
        result = handler(request)

        print(f"[TOOL END]  result : {result}")
        return result

    except Exception as e:
        print(f"[TOOL ERROR] error = {e}")
        raise

In [12]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[recommend_by_preference, load_preference, save_preference],
    middleware=[log_tool_calls],
    store=store,
)

In [13]:
user_context = UserContext(user_id="hanho")

response1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "나는 drink는 tea를 좋아하고 food는 pasta를 좋아해",
            }
        ]
    },
    context=user_context,
)

[TOOL START]  save_preference 
[TOOL ARGS] {'category': 'drink', 'value': 'tea'}
[TOOL END]  result : content='사용자 선호 정보가 저장되었습니다.' name='save_preference' tool_call_id='call_2PR2wIdPoMNH9qP6B28Dupg1'
[TOOL START]  save_preference 
[TOOL ARGS] {'category': 'food', 'value': 'pasta'}
[TOOL END]  result : content='사용자 선호 정보가 저장되었습니다.' name='save_preference' tool_call_id='call_PHPHxoC8PqKrGenNHx1y6fPN'


In [14]:
response1 = agent.invoke(
    {"messages": [{"role": "user", "content": "내가 어떤 음료를 좋아한다고 했지?"}]},
    context=user_context,
)

[TOOL START]  load_preference 
[TOOL ARGS] {'category': 'drink'}
[TOOL END]  result : content='drink에 대한 정보는 tea 입니다.' name='load_preference' tool_call_id='call_TNr6KrFXoxgeBJy91a2hyCG4'


In [15]:
from typing import TypedDict, Literal
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.tools import tool

In [16]:
class UserContext(TypedDict):
    role: Literal["user", "admin", "operator"]

In [17]:
@tool
def search_docs(query: str) -> str:
    """문서를 검색합니다."""
    return f"{query}에 대한 검색 결과입니다."


@tool
def update_db(record_id: str, new_value: str) -> str:
    """관리자만 사용할수 있는 DB수정 툴입니다."""
    return f"수정이 완료되었습니다."


@tool
def delete_file(filename: str) -> str:
    """운영자만 사용할수 있습니다. 파일을 삭제합니다."""
    return f"수정이 완료되었습니다."

In [18]:
@wrap_model_call
def restrict_tools_by_role(request, handler):

    print(request)
    role = request.runtime.context.get("role")

    new_tool = []
    if role == "user":
        new_tool = [search_docs]
    elif role == "admin":
        new_tool = [search_docs, update_db]
    elif role == "operator":
        new_tool = [search_docs, delete_file]

    new_tools = request.override(tools=new_tool)

    return handler(new_tools)

In [19]:
@wrap_tool_call
def log_tool_calls(request, handler):

    print(f"[TOOL START]  {request.tool.name} ")

    tool_call = getattr(request, "tool_call", None)
    if tool_call and "args" in tool_call and tool_call["args"] is not None:
        print(f"[TOOL ARGS] {tool_call['args']}")

    try:
        result = handler(request)

        print(f"[TOOL END]  result : {result}")
        return result

    except Exception as e:
        print(f"[TOOL ERROR] error = {e}")
        raise

In [20]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[delete_file, update_db, search_docs],
    middleware=[restrict_tools_by_role, log_tool_calls],
    context_schema=UserContext,
)

In [21]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "기밀문서.pdf 파일명의 파일을 삭제해줘"}
        ]
    },
    context={"role": "operator"},
)

ModelRequest(model=ChatOpenAI(output_version=None, profile={'name': 'GPT-5 Nano', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000285D505A2C0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000285D505BCE0>, root_client=<openai.OpenAI object at 0x00000285D5058770>, root_async_client=<openai.AsyncOpenAI object at 0x00000285D505BA80>, model_name='gpt-5-nano',

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_model
from langchain.messages import SystemMessage

In [33]:
@before_model
def force_korean(state, runtime):
    print("state : ", state)
    print(runtime)
    return {
        "messages": [SystemMessage(content="모든 답변은 반드시 한국어로 작성하세요")]
        + state["messages"]
    }

In [34]:
agent = create_agent(model="gpt-5-nano", middleware=[force_korean])

In [35]:
agent.invoke({"messages": [{"role": "user", "content": "Explain what Rag is."}]})

state :  {'messages': [HumanMessage(content='Explain what Rag is.', additional_kwargs={}, response_metadata={}, id='0a985ed7-bf9b-4dbd-8265-974eeeb1810a')]}
Runtime(context=None, store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x00000285D5200250>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f13d30d-e2b2-6447-8000-a8346fbedc87', checkpoint_ns='force_korean.before_model:cf933b11-3de4-d6d4-a85a-1685d634f050', task_id='cf933b11-3de4-d6d4-a85a-1685d634f050', thread_id=None, run_id=None, node_attempt=1, node_first_attempt_time=1776741552.0295784), server_info=None)


{'messages': [HumanMessage(content='Explain what Rag is.', additional_kwargs={}, response_metadata={}, id='0a985ed7-bf9b-4dbd-8265-974eeeb1810a'),
  SystemMessage(content='모든 답변은 반드시 한국어로 작성하세요', additional_kwargs={}, response_metadata={}, id='8b9f9c59-99f1-4008-baad-632e19c75390'),
  AIMessage(content='Rag은 일반적으로 자연어 처리(NLP) 분야에서 "Retrieval-Augmented Generation"의 약자(RAG)로 많이 사용됩니다. 이는 언어 모델이 내부에 저장된 지식만으로 답을 생성하는 것이 아니라, 외부 문서나 지식 소스를 검색(retrieval)한 정보를 근거로 답을 생성하도록 하는 방식입니다.\n\n주요 내용\n- 목표: 사실적이고 업데이트 가능한 응답을 생성하기 위해 외부 지식 소스를 활용합니다. 예를 들어 최신 기사나 특정 데이터베이스를 참조해야 하는 질문에 강점이 있습니다.\n- 구성 요소\n  - Retriever(검색기): 질문과 관련된 문서를 코퍼스에서 찾아 인덱싱된 문서들 중 후보를 선택합니다. 전통적으로는 BM25 같은 검색 기법이나 Dense Passage Retriever(DPR) 같은 벡터 기반 검색을 사용합니다.\n  - Generator(생성기): 선택된 문서들을 바탕으로 자연스러운 응답을 생성하는 시퀀스-투-시퀀스 모델입니다(T5, BART 등). 문서의 내용을 인용하거나 요약해 답에 반영합니다.\n- 작동 방법\n  1) 사용자가 질문을 입력합니다.\n  2) 검색기가 관련 문서를 찾아 후보를 제공합니다.\n  3) 생성기가 질문과 후보 문서를 함께 고려해 최종 답을 생성합니다.\n  4) 일부 구현은 문서의 어떤 부분을 어떤 방식으로 활용할지 결정하고, 필요하면 문서 인용이나